# Baseline on Colab L4
Kernel: **Colab → New Colab Server → GPU → L4**. See `colab/SETUP.md`. Never paste the HF token into a cell.

## 1. Machine (note for HARDWARE.md)

In [ ]:
!nvidia-smi; df -h /content; free -h; nproc; lscpu | grep "Model name"; python3 --version

## 2. Python 3.11 venv, code, model

In [ ]:
%%bash
set -e
cd /content
pip install -q uv
[ -d /content/venv ] || uv venv --python 3.11 /content/venv
[ -d video-indexing-yolo11 ] || git clone https://github.com/sobanali256/video-indexing-yolo11.git
cd video-indexing-yolo11 && git pull
apt-get install -y -q libgl1 libglib2.0-0
uv pip install --python /content/venv/bin/python -r requirements.txt
/content/venv/bin/python -c "import torch,av,ultralytics;print(torch.__version__,torch.cuda.is_available(),av.__version__,ultralytics.__version__)"
/content/venv/bin/python prepare.py

## 3. Hugging Face login (Colab Secret `HF_TOKEN`, else prompt)

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception:
    import getpass
    login(getpass.getpass("HF token: "))

## 4. Data: 2 files now, all 50 in the background

In [ ]:
!/content/venv/bin/hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-0000[0-1]*" --local-dir /content/finevideo
!df -h /content

In [ ]:
!nohup /content/venv/bin/hf download HuggingFaceFV/finevideo --repo-type dataset --include "data/train-000[0-4]*" --local-dir /content/finevideo > /content/download.log 2>&1 &

In [ ]:
!tail -3 /content/download.log; ls /content/finevideo/data | wc -l

## 5. Checkpoint storage: try Drive (if it fails use `--checkpoint-hf-repo` in step 6)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 6. Baseline on the subset (background). After a reset: rerun 1-5, then this cell with `--resume` added.

In [ ]:
!cd /content/video-indexing-yolo11 && nohup /content/venv/bin/python run.py --input /content/finevideo/data --output reference/ref_subset.jsonl --limit-videos 64 --dump-frames /content/drive/MyDrive/vidx/frames --checkpoint-dir /content/drive/MyDrive/vidx --run-name ref_subset > /content/baseline.log 2>&1 &

In [ ]:
!tail -5 /content/baseline.log; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## 7. Checks (after the run finishes)

In [ ]:
%%bash
cd /content/video-indexing-yolo11
PY=/content/venv/bin/python
$PY tools/check_schema.py reference/ref_subset.jsonl
$PY score.py reference/ref_subset.jsonl reference/ref_subset.jsonl
$PY tools/find_rotated.py --input /content/finevideo/data --max-files 2
$PY tools/decoder_agreement.py --input /content/finevideo/data --n 20
$PY tools/batch_agreement.py --input /content/finevideo/data --frames 1000